In [1]:
%load_ext autoreload
%autoreload 2

# `clinphen` Quickstart

This notebook walks through the full workflow a clinical researcher uses day-to-day:

1. Point a `SchemaConfig` at your hospital's physical column names (once, per environment).
2. Write / sanity-check a phenotype **standalone**, against a plain pandas `DataFrame`, with zero framework machinery.
3. Register phenotypes and run the full cohort through `FeatureMatrixBuilder` to get a tidy feature matrix.
4. Inspect the results and confirm the leakage guardrails behaved as expected.

> **Note on this environment:** real deployments use `duckdb.connect()` directly. This notebook
> runs offline, so cell 1 uses a tiny `StubConnection` that mimics the two DuckDB calls
> `clinphen` needs (`.register(name, df)` and `.execute(sql, params).fetchdf()`). Every line of
> `clinphen` itself is completely unchanged — swap in a real `duckdb.connect()` and it behaves
> identically against real Parquet/CSV files.

## 0. Set up a DuckDB connection and register the source tables

In production this is simply:
```python
import duckdb
con = duckdb.connect()
```
and you'd point `SchemaConfig.source` directly at `.parquet` file paths -- no `register()` calls needed at all.

In [15]:
"""
from duckdb_stub import StubConnection   # offline-sandbox-only shim, see note above
from build_synthetic_data import TABLES  # 3 synthetic patients -- see file for ground truth

con = StubConnection()
for name, df in TABLES.items():
    con.register(name, df)

TABLES["episodes"]
"""
import duckdb
con = duckdb.connect()

## 1. Schema configuration layer

This is the **only** place in the whole project that ever mentions a physical column name like
`OBSERVATION_RESULT_CLEAN`. Every phenotype function below only ever sees logical names
(`value`, `code`, `timestamp`, `drug`, ...). If a hospital renames a column or you move to a new
data mart, you edit this block only.

In [20]:
from icare_risk.clinphen import SchemaConfig, EpisodeTableSchema, TableSchema, DuckDBSource


path = '/app/data/mock/sirs_test'
#path = '/app/data/synthetic/2026-07-30_151244'

schema = SchemaConfig(
    episodes=EpisodeTableSchema(
        source=f"{path}/episodes.csv",              # a registered view name (or a .parquet path in prod)
        subject="SUBJECT",
        spell=None,
        encounter="ENCNTR_ID",
        admission_date="ADMISSION_DATE",
        admission_time="ADMISSION_TIME",
        discharge_date="DISCHARGE_DATE",
    ),
    domains={
        "problems": TableSchema(
            source=f"{path}/problems.csv",
            subject="SUBJECT",
            timestamp="PROBLEM_DT_TM",
            mapping={"code": "PROBLEM_CODE", "description": "PROBLEM_DESC"},
        ),
        #"prescribing": TableSchema(
        #    source="prescribing",
        #    subject="SUBJECT",
        #    timestamp="ORDER_DT_TM",
        #    mapping={"drug": "MEDICATION_NAME"},
        #),
        "vitals": TableSchema(
            source=f"{path}/vitals.csv",
            subject="SUBJECT",
            timestamp="OBSERVATION_PERFORMED_DT",
            mapping={"code": "OBSERVATION_CODE", "value": "OBSERVATION_RESULT_CLEAN"},
        ),
        "pathology": TableSchema(
            source=f"{path}/pathology.csv",
            subject="SUBJECT",
            timestamp="SAMPLE_COLLECTED_DT",
            mapping={"code": "TEST_CODE", "value": "RESULT_CLEANED"},
        ),
    },
)

source = DuckDBSource(connection=con)
schema

SchemaConfig(episodes=EpisodeTableSchema(source='/app/data/mock/sirs_test/episodes.csv', subject='SUBJECT', encounter='ENCNTR_ID', admission_date='ADMISSION_DATE', admission_time='ADMISSION_TIME', discharge_date='DISCHARGE_DATE', spell=None, mapping={}), domains={'problems': TableSchema(source='/app/data/mock/sirs_test/problems.csv', subject='SUBJECT', timestamp='PROBLEM_DT_TM', mapping={'code': 'PROBLEM_CODE', 'description': 'PROBLEM_DESC'}), 'vitals': TableSchema(source='/app/data/mock/sirs_test/vitals.csv', subject='SUBJECT', timestamp='OBSERVATION_PERFORMED_DT', mapping={'code': 'OBSERVATION_CODE', 'value': 'OBSERVATION_RESULT_CLEAN'}), 'pathology': TableSchema(source='/app/data/mock/sirs_test/pathology.csv', subject='SUBJECT', timestamp='SAMPLE_COLLECTED_DT', mapping={'code': 'TEST_CODE', 'value': 'RESULT_CLEANED'})})

## 2. Prototype a phenotype standalone (no DuckDB, no registry, no runner)

This is the core developer-experience promise of `clinphen`: you can write and debug a phenotype
against a **plain pandas DataFrame** you typed out by hand, exactly as you would in any other
notebook analysis. `EpisodeContext.from_frames(...)` applies the *same* datetime parsing and
numeric coercion used in production, so a test that passes here behaves identically once wired
into the full pipeline.

Below we hand-craft three vitals readings for a fictional patient and confirm that
`min_spo2_24h` (built-in, from `clinphen.phenotypes.windowed`) correctly **ignores** the third
reading, which falls outside the 24h window -- proving the leakage guardrail is structural, not
a matter of the phenotype author remembering to filter dates themselves.

In [17]:
import pandas as pd
from icare_risk.clinphen import EpisodeContext
from icare_risk.clinphen.phenotypes.windowed import min_spo2_24h

toy_vitals = pd.DataFrame({
    "subject": ["demo", "demo", "demo"],
    "timestamp": ["2026-01-01 01:00", "2026-01-01 20:00", "2026-01-03 09:00"],  # 3rd row is >24h later
    "code": ["SPO2", "SPO2", "SPO2"],
    "value": ["98", "91", "10"],  # the "10" must NOT affect the result -- it's out of window
})

ctx = EpisodeContext.from_frames(
    subject="demo",
    index_admission="2026-01-01 00:00",
    frames={"vitals": toy_vitals},
)

result = min_spo2_24h(ctx)
print("min_spo2_24h:", result)
assert result == 91.0, "the out-of-window reading must not leak into the result"


INSIDE FUNCTION
  subject           timestamp  code  value
0    demo 2026-01-01 01:00:00  SPO2     98
1    demo 2026-01-01 20:00:00  SPO2     91
min_spo2_24h: nan


AssertionError: the out-of-window reading must not leak into the result

### Writing your own phenotype is just as simple

Here's the entire pattern (this is literally the built-in `history_of_diabetes` implementation):

```python
@phenotype(
    name="history_of_diabetes",
    domains=["problems", "prescribing"],
    category="historical",
)
def history_of_diabetes(ctx: EpisodeContext) -> bool:
    problems = ctx.get_historical("problems", columns=["code"])
    has_icd = not problems.empty and problems["code"].str.upper().str.startswith("E11").any()

    rx = ctx.get_historical("prescribing", columns=["drug"])
    has_metformin = not rx.empty and rx["drug"].str.contains("metformin", case=False, na=False).any()

    return has_icd or has_metformin
```

Every accessor (`get_historical`, `get_current`) returns an already-sliced, already-cleaned
DataFrame -- no raw timestamp comparisons ever appear in phenotype code. The `@phenotype`
decorator is **metadata only**: it registers the function's name/required domains for
discoverability and bulk-preloading, but never wraps or alters its behaviour.

## 3. Run the full cohort through the pipeline

`FeatureMatrixBuilder` does the rest:
1. loads episodes (index admissions) via one DuckDB query,
2. works out which domain tables are needed from the registered phenotypes' `domains=[...]`,
3. bulk-preloads each required table **once** (subject-list pushed down to SQL), and
4. loops over each index episode with a leakage-safe `EpisodeContext`, running every requested phenotype in pure Pandas -- no further I/O.

In [21]:
from icare_risk.clinphen import FeatureMatrixBuilder

builder = FeatureMatrixBuilder(
    schema=schema,
    source=source,
    phenotypes=["min_spo2_24h", "min_temp_24h"],
)

feature_matrix = builder.build(raise_on_error=True)
feature_matrix

PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f543f3b3760>, domains=['vitals'], category='windowed', description='Minimum SpO2 recorded in the first 24h of the index admission.')
EEH
INSIDE FUNCTION
SPO2
PhenotypeSpec(name='min_temp_24h', func=<function min_temp_24h at 0x7f543f3b3b50>, domains=['vitals'], category='windowed', description='')
EEH
TEMP
PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f543f3b3760>, domains=['vitals'], category='windowed', description='Minimum SpO2 recorded in the first 24h of the index admission.')
EEH
INSIDE FUNCTION
SPO2
PhenotypeSpec(name='min_temp_24h', func=<function min_temp_24h at 0x7f543f3b3b50>, domains=['vitals'], category='windowed', description='')
EEH
TEMP
PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f543f3b3760>, domains=['vitals'], category='windowed', description='Minimum SpO2 recorded in the first 24h of the index admission.')
EEH
INSIDE FUNCTION
SPO2
PhenotypeSpec(name='m

,min_spo2_24h,min_temp_24h
ENCNTR_ID,,
stay_0,NaN,NaN
stay_1,115.0,26.0
stay_2,75.0,14.0
stay_3,80.0,18.0
stay_4,88.0,18.0
stay_5,105.0,16.0
stay_6,NaN,NaN
stay_7,NaN,NaN


## 4. Sanity-check against the fixtures' known ground truth

The synthetic cohort (see `build_synthetic_data.py`) deliberately includes several
**leakage traps**:

| Patient | Trap | Expected behaviour |
|---|---|---|
| P1 | Diabetes ICD code from a genuinely prior encounter | `history_of_diabetes = True` |
| P2 | Diabetes ICD code recorded **after** the index admission | `history_of_diabetes = False` |
| P3 | No ICD code, only a prior Metformin order | `history_of_diabetes = True` |
| P2 | SpO2 reading at 26.5h post-admission (outside the 24h window) | excluded from `min_spo2_24h` |
| P3 | SpO2 result stored as the messy string `"<90"` | coerced to `90.0` |
| P2 | A creatinine result recorded on day 6 (outside the 48h AKI window) | excluded from `peak_creatinine_48h` |

## Recap

- **Schema layer** (`SchemaConfig`) is the only place physical column names appear.
- **`EpisodeContext`** is the only way phenotype code touches data -- `get_historical(...)` /
  `get_current(..., window=...)` -- so leakage is prevented structurally, not by convention.
- Phenotypes are **standalone-testable**: `EpisodeContext.from_frames(...)` lets you unit-test
  or prototype in a notebook with zero DuckDB / registry / runner involvement.
- The **`@phenotype`** decorator only registers metadata (name + required domain tables) -- it
  never wraps or changes what actually runs.
- **`FeatureMatrixBuilder`** does the I/O-heavy bulk preloading once per table, then runs the
  per-patient loop entirely in memory with pure Pandas.

In [29]:
from icare_risk.clinphen.registry.registry import DEFAULT_REGISTRY
from icare_risk.clinphen.engine.runner import FeatureMatrixBuilder

# Define your configuration directly as a dictionary for instant testing
test_config = {
    "min_spo2_24h": {
        "module": "icare_risk.clinphen.phenotypes.windowed",  # Or wherever your function lives
        "function": "min_spo2_24h",
        "domains": ["vitals"],
        "kwargs": {"code": "9096705", "window": ["0h", "24h"]}
    },
    "min_temp_24h": {
        "module": "icare_risk.clinphen.phenotypes.windowed",
        "function": "min_temp_24h",
        "domains": ["vitals"],
        "kwargs": {"code": "10933766", "window": ["0h", "24h"]}
    }
}

# Register instantly from dictionary
DEFAULT_REGISTRY.from_dict(test_config)

# Run builder normally
builder = FeatureMatrixBuilder(schema=schema, source=source)
feature_matrix = builder.build()
feature_matrix


PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f5458ed3a30>, domains=['vitals'], category='config_driven', description='', kwargs={'code': '9096705', 'window': ['0h', '24h']})
INSIDE FUNCTION
Empty DataFrame
Columns: [subject, timestamp, code, value]
Index: []
PhenotypeSpec(name='min_temp_24h', func=<function min_temp_24h at 0x7f543f0332e0>, domains=['vitals'], category='config_driven', description='', kwargs={'code': '10933766', 'window': ['0h', '24h']})
TEMP
PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f5458ed3a30>, domains=['vitals'], category='config_driven', description='', kwargs={'code': '9096705', 'window': ['0h', '24h']})
INSIDE FUNCTION
   subject           timestamp      code  value
0      101 2026-09-01 12:00:00  13472364  115.0
1      101 2026-09-01 12:00:00   9096705   26.0
2      101 2026-09-01 12:00:00  10933766   39.2
PhenotypeSpec(name='min_temp_24h', func=<function min_temp_24h at 0x7f543f0332e0>, domains=['vitals'], cate

,min_spo2_24h,min_temp_24h
ENCNTR_ID,,
stay_0,NaN,NaN
stay_1,26.0,39.2
stay_2,14.0,37.1
stay_3,18.0,35.1
stay_4,18.0,37.5
stay_5,16.0,38.5
stay_6,NaN,NaN
stay_7,NaN,NaN


In [30]:
ph = DEFAULT_REGISTRY.all()
print(ph)

[PhenotypeSpec(name='min_spo2_24h', func=<function min_spo2_24h at 0x7f5458ed3a30>, domains=['vitals'], category='config_driven', description='', kwargs={'code': '9096705', 'window': ['0h', '24h']}), PhenotypeSpec(name='min_temp_24h', func=<function min_temp_24h at 0x7f543f0332e0>, domains=['vitals'], category='config_driven', description='', kwargs={'code': '10933766', 'window': ['0h', '24h']})]


In [4]:
# ---------------------------------------------------
# Generate artificial data
# ---------------------------------------------------
from icare_risk._dep.generators import generate_synthetic_cohort

# Generate cohort
print("Generating synthetic cohort...")
generated_tables = generate_synthetic_cohort(
    config_path="/app/src/icare_risk/config/icare/generate_db.yaml",
    output_dir="../data/synthetic/notebook_test"
)

Generating synthetic cohort...

📂 TARGET SAVE DIRECTORY: /app/notebooks/data/synthetic/notebook_test

⚙️ Loading standalone config from: /app/src/icare_risk/config/icare/generate_db.yaml
Generating configured tables for 100 patients...
 -> Building ICARE_EPISODES_ANON [relational]...
 -> Building ICARE_MICROBIOLOGY_ANON [relational]...
 -> Building ICARE_VITAL_SIGNS_ANON [eav_timeseries]...
 -> Building ICARE_PROBLEMS_ANON [relational]...
 -> Building ICARE_PHARMACY_PRESCRIBING_ANON [relational]...
 -> Building ICARE_PATHOLOGY_BLOOD_ANON [eav_timeseries]...


In [5]:
# 4. View the generated tables
print("\nGeneration Complete! Available tables:", list(generated_tables.keys()))

# Display the episodes table as an example
episodes_df = generated_tables.get("ICARE_VITAL_SIGNS_ANON")
if episodes_df is not None:
    display(episodes_df.head())
    print(episodes_df.OBSERVATION_NAME.unique())
    print(episodes_df.OBSERVATION_CODE.nunique())
    print(episodes_df.OBSERVATION_UNIT.nunique())

problems_df = generated_tables.get("ICARE_PROBLEMS_ANON")
problems_df.PROBLEM_CODE.unique()


Generation Complete! Available tables: ['ICARE_EPISODES_ANON', 'ICARE_MICROBIOLOGY_ANON', 'ICARE_VITAL_SIGNS_ANON', 'ICARE_PROBLEMS_ANON', 'ICARE_PHARMACY_PRESCRIBING_ANON', 'ICARE_PATHOLOGY_BLOOD_ANON']


,SUBJECT,ENCNTR_ID,OBSERVATION_CODE,OBSERVATION_NAME,OBSERVATION_PERFORMED_DT,OBSERVATION_START_DT,OBSERVATION_END_DT,OBSERVATION_RESULT_CLEAN,OBSERVATION_UNIT
1095,10001,6215401,104232993,cpap,2022-07-12 20:00:00,2022-07-12 20:00:00,2022-07-12 20:00:00,104.66,none
1371,10001,6215401,104232993,cpap,2022-07-13 08:00:00,2022-07-13 08:00:00,2022-07-13 08:00:00,143.98,none
2935,10001,6215401,104232993,cpap,2022-07-16 04:00:00,2022-07-16 04:00:00,2022-07-16 04:00:00,36.99,none
3119,10001,6215401,104232993,cpap,2022-07-16 12:00:00,2022-07-16 12:00:00,2022-07-16 12:00:00,287.50,none
4407,10001,6215401,104232993,cpap,2022-07-18 20:00:00,2022-07-18 20:00:00,2022-07-18 20:00:00,202.58,none


['cpap' 'bipap' 'respiratory support device (itu)' 'temperature'
 'mean arterial pressure, cuff' 'weight measured' 'height/length measured'
 'alanine aminotransferase level, blood'
 'brain natriuretic peptide level, blood'
 'c-reactive protein level, blood' 'creatinine level, blood'
 'ferritin level, blood' 'hco3 cap' 'lactate dehydrogenase level, blood'
 'lymphocyte count, blood' 'ferritin' 'oxygen flow rate' 'oxygen therapy'
 'mean arterial pressure, invasive' 'ipap' 'epap' 'tidal volume'
 'peak airway pressure' 'oxygen tubing & mask' 'fio2 level poc'
 'systolic blood pressure sitting' 'diastolic blood pressure sitting'
 'height & weight' 'news blood pressure lying' 'oxygen saturation'
 'frailty scale category' 'patient on oxygen' 'oxygen amount'
 'fio2 - delivered' 'systolic blood pressure cuff'
 'diastolic blood pressure cuff' 'poct base excess' 'heart rate'
 'systolic blood pressure standing' 'diastolic blood pressure standing'
 'troponin i (ng/l) level, blood' 'd-dimer (ug/l) lev

array(['38341003', '84114007', '40930008', '44054006', '77386006',
       '19030005', '13645005', '73211009', '46635009', '414545008',
       '197321007', '709044004'], dtype=object)

In [3]:
# ---------------------------------------------------
# Derive phenotypes
# ---------------------------------------------------
# Libraries
import duckdb

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.registry.registry import DEFAULT_REGISTRY
from icare_risk.clinphen.registry.registry import register_from_yaml
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.engine.runner import FeatureMatrixBuilder

# Define paths
schema_cfg = '/app/src/icare_risk/config/icare/schema.yaml'
phenotype_cfg = '/app/src/icare_risk/config/icare/phenotypes.yaml'

# Load schema definition
schema = load_schema_from_yaml(schema_cfg)

# Create connection source
source = DuckDBSource(connection=duckdb.connect())

# Load the phenotypes definitions
DEFAULT_REGISTRY._specs.clear() # Clear before running
register_from_yaml(phenotype_cfg, DEFAULT_REGISTRY)

# Create matrix
builder = FeatureMatrixBuilder(
    schema=schema, source=source,
    phenotypes=['has_diabetes']
)

feature_matrix = builder.build(raise_on_error=True)
display(feature_matrix)

print(feature_matrix.has_diabetes.sum())
print(feature_matrix.shape())

ValueError: Empty module name